<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c12-header.png" alt="Nextia Learning · Fine-Tuning and Adapting AI Models" width="100%">

# Solution: Should this team fine-tune?

**[Should this team fine-tune?](https://learning.nextia-ai.com/courses/fine-tuning/m07/should-this-team-fine-tune/)** · Fine-Tuning and Adapting AI Models · Module 7, case study · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** decide, with evidence, whether a small team should fine-tune a small model. You measure a prompt-only system, word counts and a small model on the same held-out messages, compare two recorded LoRA adapters under one measure, look at the holdout cases once, and write a go/no-go memo.

| | |
|---|---|
| **Time** | About 2.5 hours with the case study page. The notebook runs in about 1 minute: the model replies and the training runs are recorded. |
| **Needs** | An internet connection for the setup (about 7 MB). No account, no key, no graphics card. The optional model cells download the small model (about 270 MB) and need about 2 GB of free memory. |
| **You should know** | The whole course, most of all [Define success and constraints](https://learning.nextia-ai.com/courses/fine-tuning/m01/define-success-and-constraints/), [Full and parameter-efficient tuning](https://learning.nextia-ai.com/courses/fine-tuning/m03/full-and-parameter-efficient-tuning/) and [Compare baselines](https://learning.nextia-ai.com/courses/fine-tuning/m05/compare-baselines/). |
| **Data** | CLINC150 (Larson et al., 2019), CC BY 3.0: the 15 intents of its travel domain and its out-of-scope queries. |
| **Tested** | Python 3.12 with pandas 3.0, scikit-learn 1.9.1 and matplotlib, in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*). The optional model cells: PyTorch 2.14.1 (CPU), transformers 5.19.0, peft 0.21.2. Colab and Kaggle are not tested |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/fine-tuning/m07/should-this-team-fine-tune/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C12/should-this-team-fine-tune/should-this-team-fine-tune-solution.ipynb).

## Setup: packages and data

Run the next cell. It checks the packages, downloads **CLINC150** (2.5 MB) from its publisher's GitHub repository at one fixed version, and the case study's recorded runs (about 4 MB) from the Nextia Learning labs repository. It checks every file's SHA-256 and puts everything in a new folder `fernway`. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: CLINC150 with 23,700 queries, and 15 recorded files.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `fernway` (in Colab: the **Files** panel on the left), then run the cell again.

Data: "CLINC150" (An Evaluation Dataset for Intent Classification and Out-of-Scope Prediction) by Larson et al., 2019, https://github.com/clinc/oos-eval, licensed under CC BY 3.0. This notebook uses a sample of the travel domain and the out-of-scope queries; see `dataset.md`.

In [ ]:
import hashlib, json, os, statistics, subprocess, urllib.request
from pathlib import Path
import pandas as pd
import sklearn

CLINC_URL = "https://raw.githubusercontent.com/clinc/oos-eval/828f8093932c8fe6ca7936c3d2e52903b1c523de/data/data_full.json"
CLINC_SHA256 = "36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0"
RUNS_URL = os.environ.get("NX_RUNS_URL", "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C12/should-this-team-fine-tune/runs/")
RUN_FILES = {
    "hosted.jsonl": "a48426d1076a5cc7fc0a931d9f101e280efabcdad38693c4d8fb2c482ab67ac7",
    "predictions/words.jsonl": "bec091817d31019ae7f342121449d6f5a12478e3df50a16d81a05e2c0c7f5c92",
    "predictions/words-full.jsonl": "5b55e5ad747ad7c3284ed8e70c80bc755ec1ff54fde06b89de2cd39916a84ffa",
    "predictions/base-short.jsonl": "8a5cf8c97d3018258aa390a8bd55b0d386593b10a3a2cf158357088dc6b25365",
    "predictions/base-choice.jsonl": "2a60919a6453c2097b6539bc9e630e97ab4b8492da13b93d89b6cae166dd8e47",
    "predictions/adapter.jsonl": "d654c489d2f35a6cd0d51e9552226de97bcf4d69128895f3f58d0ec95390c9dd",
    "predictions/adapter-full.jsonl": "a0adfab207ca9f0b32a96fb77ae5b73a331b9db95920295133535f956f449830",
    "lora-r8/log.jsonl": "d54be41bf66c0b6f134fbc24690eaa3b5a32353e369db5c32bbfb71a9d90dc0f",
    "lora-r8/summary.json": "600887cd7f212dcce7a933436060405113ea60917fb333a71e66d763d13a1151",
    "lora-r8/adapter/adapter_config.json": "f54995d6df3ca961c08d61feeb4715931fe626d646b6aba5f99c3dfa9d582756",
    "lora-r8/adapter/adapter_model.safetensors": "bae18dda7f35edd0fa47256a37270e8877a6c02c8d5b5ec8e419dd304bbdb6df",
    "lora-r8-full/log.jsonl": "e2122b09c7e01fdbed9ae41aeccf984526ec638edaa8ff6296525bb59b59da79",
    "lora-r8-full/summary.json": "3c2d261c54b48f72b565d7f026b49ee85cab8780852ea3f8157538b93b12f5a7",
    "lora-r8-full/adapter/adapter_config.json": "b68246800fd8884727bc4ba43a714ae48bf9e68eb6770b72dbf3026561c0e023",
    "lora-r8-full/adapter/adapter_model.safetensors": "39b7a701dff74536b924fbeb90dfa921f623eb0af4e4c507badda8e14f9043c7",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HOME = Path("fernway").resolve()
HOME.mkdir(exist_ok=True)
(HOME / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def get(url, path, expected, local=None):
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        if local is not None and local.exists():   # a copy next to the notebook (a local clone of the labs)
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(local.read_bytes())
        else:
            download(url, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{path.name} has the wrong checksum. Delete the folder fernway and run again.")

get(CLINC_URL, HOME / "data_full.json", CLINC_SHA256)
for name, expected in RUN_FILES.items():
    get(RUNS_URL + name, HOME / "runs" / name, expected, Path("runs") / name)
os.chdir(HOME)
raw = json.loads(Path("data_full.json").read_text(encoding="utf-8"))
print(f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}")
print(f"Ready: CLINC150 with {sum(len(v) for v in raw.values()):,} queries, and {len(RUN_FILES)} recorded files.")

## Setup: the case study's code

The next cell defines the case study's code: the 15 intents, the fixed sample, the near-copy check, the answer format with its strict check, and the one measure, `score()`. It is the same code that made the page's numbers. You do not need to read it now: each part is explained where it is used. Run the cell. It prints nothing.

In [ ]:
import hashlib
import json
import random
import re
from pathlib import Path

COMMIT = "828f8093932c8fe6ca7936c3d2e52903b1c523de"
URL = f"https://raw.githubusercontent.com/clinc/oos-eval/{COMMIT}/data/data_full.json"
SHA256 = "36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0"

# The travel domain of CLINC150 (its data/domains.json), sorted, then the out-of-scope label.
INTENTS = (
    "book_flight", "book_hotel", "car_rental", "carry_on", "exchange_rate", "flight_status", "international_visa",
    "lost_luggage", "plug_type", "timezone", "translate", "travel_alert", "travel_notification",
    "travel_suggestion", "vaccines",
)
OUT = "out_of_scope"
LABELS = INTENTS + (OUT,)

# The sample: per intent from CLINC's own splits; out-of-scope queries from its oos_* lists.
SEED = 7
SAMPLE = {  # part: (CLINC split, per intent, out-of-scope split, out-of-scope count)
    "train": ("train", 50, "oos_train", 100),
    "val": ("val", 4, "oos_val", 20),
    "dev": ("val", 10, "oos_val", 50),
    "holdout": ("test", 10, "oos_test", 50),
}

# The short instruction: the adapter is trained with it, and the base model is tried with it.
SHORT = (
    "You sort messages sent to the help chat of Fernway, a travel app. Reply with JSON only, in this form: "
    '{"intent": NAME}. NAME is one of: ' + ", ".join(LABELS) + "."
)

# The prompt-only system: the same task with a description of every intent (no examples, no training).
DESCRIPTIONS = {
    "book_flight": "find or book a flight",
    "book_hotel": "find or book a hotel room",
    "car_rental": "rent a car",
    "carry_on": "rules for carry-on (cabin) bags",
    "exchange_rate": "currency exchange rates",
    "flight_status": "the status of a flight: on time, delayed, landed",
    "international_visa": "whether a visa is needed for a country",
    "lost_luggage": "luggage that the airline lost",
    "plug_type": "power plugs and adapters in another country",
    "timezone": "the time zone of a place",
    "translate": "how to say something in another language",
    "travel_alert": "travel warnings or alerts for a country",
    "travel_notification": "the customer tells the dates and places of a trip, so that the card works there",
    "travel_suggestion": "ideas for things to do or see on a trip",
    "vaccines": "vaccines or shots needed before a trip",
}
PROMPT = (
    "You sort messages sent to the help chat of Fernway, a travel app. The app can handle these 15 intents:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DESCRIPTIONS.items())
    + "\n\nIf the message asks for anything else, the intent is out_of_scope: a person answers it.\n"
    'Reply with JSON only, in this form: {"intent": NAME}. NAME is one of the 15 intents or out_of_scope.'
)


def format_answer(intent: str) -> str:
    """The one way to write an answer: {"intent": "book_flight"}."""
    return json.dumps({"intent": intent})


def parse_answer(raw: str) -> tuple[str | None, str]:
    """(intent, "") when the reply is exactly one valid answer, else (None, the problem)."""
    text = (raw or "").strip()
    if not text:
        return None, "empty reply"
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        try:
            value, end = json.JSONDecoder().raw_decode(text)
        except json.JSONDecodeError:
            return None, "not JSON"
        if text[end:].strip():
            return None, "JSON with extra text"
    if not isinstance(value, dict):
        return None, "not a JSON object"
    if set(value) != {"intent"}:
        return None, "wrong keys"
    if value["intent"] not in LABELS:
        return None, "unknown intent"
    return value["intent"], ""


def sha256(path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def normalize(text: str) -> str:
    return " ".join(re.findall(r"[a-z0-9']+", text.lower()))


def shingles(text: str, n: int = 3) -> frozenset:
    words = normalize(text).split()
    return frozenset(tuple(words[i:i + n]) for i in range(max(1, len(words) - n + 1)))


def overlap(a: frozenset, b: frozenset) -> float:
    return len(a & b) / len(a | b) if a and b else 0.0


def build_sample(raw: dict) -> dict:
    """The four parts, each a list of {"id", "text", "intent"}; fixed by SEED. dev and val never share a query."""
    rng = random.Random(SEED)
    parts = {name: [] for name in SAMPLE}
    used = set()  # (CLINC split, index): dev and val both come from CLINC's val split
    for name, (split, per_intent, oos_split, n_oos) in SAMPLE.items():
        for intent in INTENTS:
            pool = [i for i, (_, label) in enumerate(raw[split]) if label == intent and (split, i) not in used]
            for i in sorted(rng.sample(pool, per_intent)):
                used.add((split, i))
                parts[name].append({"id": f"{split}-{i:04d}", "text": raw[split][i][0], "intent": intent})
        pool = [i for i in range(len(raw[oos_split])) if (oos_split, i) not in used]
        for i in sorted(rng.sample(pool, n_oos)):
            used.add((oos_split, i))
            parts[name].append({"id": f"{oos_split}-{i:04d}", "text": raw[oos_split][i][0], "intent": OUT})
    return parts


def build_full(raw: dict) -> list[dict]:
    """Every travel query of CLINC's train split (100 per intent) and its 100 out-of-scope training queries:
    the "more data" run. It holds every record of the 50-per-intent sample, and 750 more."""
    rows = [{"id": f"train-{i:04d}", "text": t, "intent": label} for i, (t, label) in enumerate(raw["train"])
            if label in INTENTS]
    return rows + [{"id": f"oos_train-{i:04d}", "text": t, "intent": OUT} for i, (t, _) in enumerate(raw["oos_train"])]


# New messages, written for this case study (constructed, not from CLINC150), with the label a person at
# Fernway would give. Used once, at the end, for the end-to-end run.
NEW = [
    ("my flight to lisbon tomorrow, is it still on time?", "flight_status"),
    ("do i need a visa for vietnam with a uk passport", "international_visa"),
    ("the airline lost my suitcase in frankfurt, what do i do", "lost_luggage"),
    ("how do you say where is the train station in japanese", "translate"),
    ("can i bring a guitar as a carry on with delta", "carry_on"),
    ("is it safe to travel to egypt right now", "travel_alert"),
    ("find me a cheap room in berlin for 3 nights from june 4", "book_hotel"),
    ("i want a refund for the hotel you booked, it was dirty", OUT),
    ("my card was charged twice for the same booking", OUT),
    ("can you recommend a good dentist", OUT),
]


def leak_check(parts: dict, near: float = 0.4) -> dict:
    """Training or validation queries that equal or nearly copy a dev or holdout query (word 3-grams)."""
    held = [(c, shingles(c["text"])) for p in ("dev", "holdout") for c in parts[p]]
    found = {}
    for p in ("train", "val"):
        for r in parts[p]:
            sh = shingles(r["text"])
            best = max(((overlap(sh, s), c) for c, s in held), key=lambda x: x[0])
            if best[0] >= near:
                found[r["id"]] = {"part": p, "text": r["text"], "intent": r["intent"], "copy_of": best[1]["id"],
                                  "copy_text": best[1]["text"], "overlap": round(best[0], 2)}
    return found


def to_record(row: dict) -> dict:
    """A training record in the chat format of the course's project (adapt/train.py reads `messages`)."""
    return {"id": row["id"], "intent": row["intent"],
            "messages": [{"role": "system", "content": SHORT}, {"role": "user", "content": row["text"]},
                         {"role": "assistant", "content": format_answer(row["intent"])}]}


def score(cases: list[dict], replies: dict) -> dict:
    """The case study's one measure. replies: case id -> raw reply. No valid answer = not sent to a person.

    answers          valid answers (strict JSON, a known intent)
    intent_right     in-scope cases with the right intent (an answer of out_of_scope counts as wrong)
    out_caught       out-of-scope cases answered out_of_scope (they reach a person)
    false_handovers  in-scope cases answered out_of_scope (a person gets a message the app could handle)
    decisions_right  cases whose answer is exactly right: the intent, or out_of_scope for out-of-scope cases
    """
    n_in = sum(c["intent"] != OUT for c in cases)
    n_out = len(cases) - n_in
    answers = intent_right = out_caught = false_handovers = right = 0
    for c in cases:
        got, _ = parse_answer(replies.get(c["id"], ""))
        answers += got is not None
        if c["intent"] == OUT:
            out_caught += got == OUT
        else:
            intent_right += got == c["intent"]
            false_handovers += got == OUT
        right += got == c["intent"]
    return {"cases": len(cases), "in_scope": n_in, "out_of_scope": n_out, "answers": answers,
            "intent_right": intent_right, "out_caught": out_caught, "false_handovers": false_handovers,
            "decisions_right": right}


def line(s: dict) -> str:
    return (f"answers {s['answers']}/{s['cases']} | intent {s['intent_right']}/{s['in_scope']} | "
            f"out of scope: caught {s['out_caught']}/{s['out_of_scope']}, false handovers {s['false_handovers']} | "
            f"decisions {s['decisions_right']}/{s['cases']}")


def read_jsonl(path) -> list[dict]:
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]


def write_jsonl(path, rows) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")

## Setup: check helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

Fernway's travel app has a help chat. It recognises 15 kinds of request, the **intents**, and starts the matching flow. Any other message is **out of scope** and must reach a person. Teo, the mobile engineer, wants a small model inside the app, so that the chat also works offline. Hana, the data scientist, must decide with evidence.

CLINC150 has 150 intents in 10 domains, and queries that fit none of them. The case study uses the travel domain. First, look at its size.

> **Predict.** How many training queries does each travel intent have?

In [ ]:
travel = [label for _, label in raw["train"] if label in INTENTS]
print({part: len(rows) for part, rows in raw.items()})
print(len(set(travel)), "travel intents;", "training queries per intent:", sorted(set(pd.Series(travel).value_counts())))
print([text for text, label in raw["oos_train"][:3]])

> **Check.** You should see 15,000 train queries, 100 in `oos_train`, `15 travel intents; training queries per intent: [100]`, and three out-of-scope queries that have nothing to do with travel.

Now draw the fixed sample, and remove the training and validation queries that nearly copy a dev or holdout query (word 3-gram overlap of 0.4 or more, the course's rule from [Clean, split and check for leakage](https://learning.nextia-ai.com/courses/fine-tuning/m02/clean-split-and-check-for-leakage/)).

In [ ]:
parts = build_sample(raw)
leaks = leak_check(parts)
for p in ("train", "val"):
    parts[p] = [r for r in parts[p] if r["id"] not in leaks]
for name, rows in parts.items():
    print(f"{name:8s} {len(rows):4d} queries, out of scope {sum(r['intent'] == OUT for r in rows)}")
print("near copies removed:", len(leaks))
for v in sorted(leaks.values(), key=lambda v: -v["overlap"])[:3]:
    print(f"  {v['overlap']}  {v['text']!r}  ~  {v['copy_text']!r}")

> **Check.** You should see train 813 (out of scope 100), val 72, dev 200, holdout 200 (out of scope 50 each), and `near copies removed: 45`. The closest pair differs in one word: the crowd workers paraphrased the same seed phrases.

**Dev** is for every comparison and choice. **Holdout** stays closed until section 8.

## 2. One measure for every system

Every system answers with one JSON object, `{"intent": "book_flight"}`, and `parse_answer()` accepts nothing else. `score()` counts the decisions that matter to Fernway. An invalid reply counts as "not sent to a person", because nobody reads it.

> **Predict.** Four replies to four dev cases follow. How many are valid, and how many decisions are right?

In [ ]:
tiny_cases = [{"id": "a", "text": "", "intent": "book_flight"}, {"id": "b", "text": "", "intent": "vaccines"},
              {"id": "c", "text": "", "intent": OUT}, {"id": "d", "text": "", "intent": OUT}]
tiny_replies = {"a": '{"intent": "book_flight"}', "b": '{"intent": "out_of_scope"}',
                "c": 'Sure! {"intent": "out_of_scope"}', "d": '{"intent": "out_of_scope"}'}
for case_id, reply in tiny_replies.items():
    print(case_id, parse_answer(reply))
print(line(score(tiny_cases, tiny_replies)))

> **Check.** You should see `answers 3/4 | intent 1/2 | out of scope: caught 1/2, false handovers 1 | decisions 2/4`. Reply c has the right idea, but extra text: invalid, so that message reaches nobody. Reply b sends an in-scope message to a person: a **false handover**.

## 3. What Fernway has, and the cheap alternatives

The recorded replies are in `runs/`. `replies(system, split)` reads them. The hosted system is chat-small with a prompt that describes the 15 intents: **prompt only**, no training.

In [ ]:
def replies(system, split):
    path = Path("runs/hosted.jsonl" if system == "hosted" else f"runs/predictions/{system}.jsonl")
    return {r["id"]: r for r in map(json.loads, path.read_text(encoding="utf-8").splitlines()) if r["split"] == split}

dev = parts["dev"]
hosted = replies("hosted", "dev")
print(line(score(dev, {k: r["raw"] for k, r in hosted.items()})))
print("median seconds:", statistics.median(r["seconds"] for r in hosted.values()))
cost = sum(r["tokens_in"] * 0.10 + r["tokens_out"] * 0.50 for r in hosted.values()) / 1e6
print(f"US${1000 * cost / len(hosted):.3f} per 1,000 messages (list price, US$0.10 / 0.50 per 1M tokens)")

> **Check.** You should see `answers 200/200 | intent 150/150 | out of scope: caught 48/50, false handovers 0 | decisions 198/200`, a median of about 1.11 seconds (network included), and US$0.043 per 1,000 messages. Prices change: check the current price page.

The cheapest local alternative is **word counts**: TF-IDF of words and word pairs, and logistic regression, trained on the same 813 records that the adapter learns from. `out_of_scope` is a 16th class.

> **Predict.** Will word counts catch more or fewer out-of-scope messages than the hosted model?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def fit_words(rows):
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(C=10, max_iter=2000))
    return model.fit([r["text"] for r in rows], [r["intent"] for r in rows])

words = fit_words(parts["train"])
words_replies = {c["id"]: format_answer(p) for c, p in zip(dev, words.predict([c["text"] for c in dev]))}
print(line(score(dev, words_replies)))

> **Check.** You should see `answers 200/200 | intent 141/150 | out of scope: caught 38/50, false handovers 6 | decisions 179/200`. It is fast and free, but it sends 12 out-of-scope messages to an automatic flow.

The small model, SmolLM2-135M-Instruct, without training, was recorded twice: with the short instruction that the adapter is trained with (`base-short`), and with the hosted prompt and the answer **chosen** from the 16 valid answers by its probability (`base-choice`, constrained output).

In [ ]:
for system in ("base-short", "base-choice"):
    print(f"{system:12s}", line(score(dev, {k: r["raw"] for k, r in replies(system, "dev").items()})))
print(repr(replies("base-short", "dev")[dev[0]["id"]]["raw"]))

> **Check.** You should see `answers 1/200 | intent 1/150 | out of scope: caught 0/50, false handovers 0 | decisions 1/200` and `answers 200/200 | intent 67/150 | out of scope: caught 43/50, false handovers 55 | decisions 110/200`. The short instruction gives prose. The fixed choice is always valid, but it answers `out_of_scope` for almost every second message: it catches many out-of-scope messages only because it hands over 55 that the app could answer.

## 4. Targets, before any training

Hana wrote these targets before training (the page has the memo). The stop rule is `out_caught`: a message the app cannot handle must reach a person.

In [ ]:
TARGETS = {"valid_answers": 0.99, "intent_right": 0.95, "out_caught": 0.90, "false_handovers": 0.05, "median_seconds": 0.5}

def measured(s, seconds):
    return {"valid_answers": s["answers"] / s["cases"], "intent_right": s["intent_right"] / s["in_scope"],
            "out_caught": s["out_caught"] / s["out_of_scope"], "false_handovers": s["false_handovers"] / s["in_scope"],
            "median_seconds": seconds}

def report(s, seconds):
    for name, value in measured(s, seconds).items():
        ok = value <= TARGETS[name] if name in ("false_handovers", "median_seconds") else value >= TARGETS[name]
        print(f"  {'met    ' if ok else 'NOT MET'}  {name:16s} target {TARGETS[name]:<5}  measured {value:.3f}")

report(score(dev, {k: r["raw"] for k, r in hosted.items()}), statistics.median(r["seconds"] for r in hosted.values()))

> **Check.** The hosted system meets four targets and misses only the median time (1.11 s against 0.5 s). That is Teo's reason: speed, offline use, cost and privacy, not quality.

## 5. The adapter: two recorded runs

Hana trained one LoRA adapter (rank 8 on the query and value layers) with the course's recipe, unchanged: 2 epochs, learning rate 2e-4, batch 8 ([Plan and configure the run](https://learning.nextia-ai.com/courses/fine-tuning/m04/plan-and-configure-the-run/)). Teo then asked for more data, so a second run changed one thing: every travel query of CLINC's train split, 100 per intent instead of 50. Read the two runs' summaries and curves.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.2))
for run, style in (("lora-r8", "-"), ("lora-r8-full", "--")):
    summary = json.loads(Path(f"runs/{run}/summary.json").read_text())
    log = [json.loads(x) for x in Path(f"runs/{run}/log.jsonl").read_text().splitlines()]
    val = [(r["step"], r["val_loss"]) for r in log if "val_loss" in r]
    ax.plot(*zip(*val), style, label=f"{run}: validation loss")
    print(f"{run:13s} {summary['train_records']} records, {summary['steps']} steps, {summary['train_seconds']:.0f} s of steps,"
          f" peak {summary['peak_mb']:.0f} MB, best validation loss {summary['best_val_loss']} at step {summary['best_step']}")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); plt.show()

> **Check.** You should see `lora-r8` with 813 records and 204 steps (best validation loss 0.0515), and `lora-r8-full` with 1530 records and 384 steps (0.0241). Both stayed under 1.8 GB. The times were measured on the course's computer while other jobs ran: yours will differ.

> **Optional: train again.** Training needs the course's project `ticket-adapter` (Module 4 or later) on your computer, the small model (about 270 MB) and about 4 to 8 minutes of CPU. The notebook does not need it: the replies of both adapters are recorded. The next cell writes the training files and the configuration into `fernway/export/`; the page's section *Train it yourself* has the steps.

In [ ]:
CONFIG = """# The case study's adapter: the course's recipe (configs/lora-r8.toml), with Fernway's data.
name = "fernway-r8"
seed = 12
threads = 2

[data]
train = "data/fernway/train.jsonl"
val = "data/fernway/val.jsonl"
max_tokens = 320

[lora]
rank = 8
alpha = 16
dropout = 0.05
targets = ["q_proj", "v_proj"]

[train]
learning_rate = 2e-4
batch_size = 8
epochs = 2
eval_every = 25
checkpoint_every = 50
gradient_checkpointing = true
keep_checkpoints = 2
"""
export = Path("export")
for name in ("train", "val"):
    write_jsonl(export / "data" / "fernway" / f"{name}.jsonl", [to_record(r) for r in parts[name]])
(export / "configs").mkdir(exist_ok=True)
(export / "configs" / "fernway-r8.toml").write_text(CONFIG, encoding="utf-8")
print(sorted(str(p.relative_to(export)) for p in export.rglob("*.*")))
print(hashlib.sha256((export / "data/fernway/train.jsonl").read_bytes()).hexdigest()[:12])

> **Check.** You should see three files, and `ab8807de334c`: the same training file, byte for byte, as the recorded run used.

## 6. A fair comparison on dev

Every system, the same 200 dev cases, the same `score()`. `words-full` is the word-count model trained on the same 100 per intent as `lora-r8-full`, so that each adapter has a fair cheap rival.

> **Predict.** Which local system catches the most out-of-scope messages without many false handovers?

In [ ]:
full = build_full(raw)   # every travel training query, 100 per intent, and the 100 out-of-scope ones
full_leaks = leak_check({"train": full, "val": [], "dev": parts["dev"], "holdout": parts["holdout"]})
words_full = fit_words([r for r in full if r["id"] not in full_leaks])
rows = []
for system in ("hosted", "words", "words-full", "base-choice", "adapter", "adapter-full"):
    rec = replies(system, "dev")
    s = score(dev, {k: r["raw"] for k, r in rec.items()})
    rows.append({"system": system, **{k: s[k] for k in ("answers", "intent_right", "out_caught", "false_handovers", "decisions_right")},
                 "median_s": round(statistics.median(r["seconds"] for r in rec.values()), 3)})
table = pd.DataFrame(rows).set_index("system")
table

> **Check.** `adapter-full` has the best local decisions (183/200) and the right intent on 143 of 150, but it catches only 40 of 50 out-of-scope messages; the first adapter caught 41. More in-scope data made the intents better and the stop rule no better. The word counts show the same pattern: 38 caught with 50 per intent, 33 with 100.

> **Your turn 1.** Put the share of out-of-scope dev messages that `adapter-full` catches (a number from 0 to 1) in `caught_share`. Then run the check cell.

In [ ]:
s = score(dev, {k: r["raw"] for k, r in replies("adapter-full", "dev").items()})
caught_share = s["out_caught"] / s["out_of_scope"]
print(caught_share)

In [ ]:
expect('caught_share', caught_share if caught_share is ... else round(caught_share, 3), 0.8, tolerance=0.001)

Now look at the out-of-scope messages that `adapter-full` sent to an automatic flow.

In [ ]:
for c in dev:
    got, _ = parse_answer(replies("adapter-full", "dev")[c["id"]]["raw"])
    if c["intent"] == OUT and got != OUT:
        print(f"{got:20s} <- {c['text']}")

> **Check.** You should see 10 messages, such as "how to unclog a drain" answered `plug_type`. A classifier picks the nearest intent it knows. The adapter saw only 100 out-of-scope examples, against 1,430 travel ones.

> **Your turn 2.** Write `missed_targets(s, seconds)`: it returns the sorted list of the names in `TARGETS` that a system does **not** meet. Use `measured()`. Then run the check cell, which tests it on `adapter-full`.

In [ ]:
def missed_targets(s, seconds):
    values = measured(s, seconds)
    lower_is_better = ("false_handovers", "median_seconds")
    return sorted(n for n, v in values.items()
                  if (v > TARGETS[n] if n in lower_is_better else v < TARGETS[n]))

In [ ]:
expect_hash('missed_targets for adapter-full', missed_targets(score(dev, {k: r["raw"] for k, r in replies("adapter-full", "dev").items()}),
               statistics.median(r["seconds"] for r in replies("adapter-full", "dev").values())), '9aa5e4707bdadbb1')

## 7. Change one thing: a confidence threshold

A cheap patch for the word counts: when the model gives its best intent less than a **threshold** of probability, send the message to a person. ([Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/) in the machine learning course.)

> **Predict.** As the threshold rises from 0 to 0.5, what happens to the out-of-scope messages caught, and to the false handovers?

In [ ]:
probs, labels = words.predict_proba([c["text"] for c in dev]), list(words.classes_)
for threshold in (0.0, 0.2, 0.3, 0.35, 0.4, 0.5):
    preds = [labels[p.argmax()] if p.max() >= threshold else OUT for p in probs]
    print(f"threshold {threshold:4}:", line(score(dev, {c["id"]: format_answer(x) for c, x in zip(dev, preds)})))

> **Check.** At 0.35 you should see `46/50` caught with 14 false handovers; at 0.5, all 50 caught with 24 false handovers. Every out-of-scope message caught costs in-scope messages handed over. No threshold meets both targets (at least 45 caught, at most 7 false handovers).

<details><summary>A failure case: count an invalid reply as "sent to a person"</summary>

A tempting shortcut: "if the model's reply is not valid, send the message to a person". Then `base-short`, which writes prose almost every time, "catches" all 50 out-of-scope messages, and hands over 149 of 150 in-scope ones. The shortcut hides a broken model behind a safe-looking number. Keep "no valid answer = not sent to a person" in the measure, and handle invalid replies as a separate failure.

</details>

## 8. The final test, once

The decision on dev is already no-go: no local system meets the stop rule. Hana still looks at the holdout cases once, to report numbers that no choice has touched.

In [ ]:
holdout = parts["holdout"]
for system in ("hosted", "words", "words-full", "base-choice", "adapter", "adapter-full"):
    print(f"{system:13s}", line(score(holdout, {k: r["raw"] for k, r in replies(system, "holdout").items()})))

> **Check.** `hosted`: `answers 200/200 | intent 149/150 | out of scope: caught 50/50, false handovers 1 | decisions 199/200`. `adapter-full`: `answers 199/200 | intent 140/150 | out of scope: caught 36/50, false handovers 5 | decisions 176/200`. The holdout cases tell the same story as dev.

## 9. End to end: new messages

Ten messages written for this case study (not from CLINC150), with the label that a person at Fernway would give. The word counts run here; the adapter's and the hosted replies are recorded. To run the adapter yourself, set `RUN_MODEL = True`: it installs PyTorch, transformers and peft at the course project's versions (about 700 MB) and downloads the small model (about 270 MB).

In [ ]:
RUN_MODEL = False
new = [{"id": f"new-{i:02d}", "text": t, "intent": label} for i, (t, label) in enumerate(NEW, 1)]
words_new = dict(zip([c["id"] for c in new], words_full.predict([c["text"] for c in new])))
adapter_new = {k: r["raw"] for k, r in replies("adapter-full", "new").items()}
hosted_new = {k: r["raw"] for k, r in replies("hosted", "new").items()}
for c in new:
    print(f"{c['text'][:48]:48s} {c['intent']:19s} words {words_new[c['id']]:19s} "
          f"adapter {parse_answer(adapter_new[c['id']])[0] or 'invalid':19s} hosted {parse_answer(hosted_new[c['id']])[0]}")

> **Check.** The first column is the right label. Look at the last three messages: a refund request, a double charge, a dentist. Each one must reach a person.

In [ ]:
ADAPTER = "runs/lora-r8-full/adapter"   # your own run: the full path of runs/fernway-r8/adapter
RECORDED = "adapter-full"                # your own run uses the first recipe: "adapter"
if RUN_MODEL:
    # The course project's pinned versions; on Windows and Linux, install torch first from
    # https://download.pytorch.org/whl/cpu (the CPU-only build)
    %pip install -q --disable-pip-version-check torch==2.14.1 transformers==5.19.0 peft==0.21.2 tokenizers==0.23.3 safetensors==0.8.0
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    torch.set_num_threads(2)
    MODEL, REVISION = "HuggingFaceTB/SmolLM2-135M-Instruct", "12fd25f77366fa6b3b4b768ec3050bf629380bac"
    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
    base = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.float32, attn_implementation="sdpa")
    model = PeftModel.from_pretrained(base, ADAPTER).eval()   # float32, as in the recordings
    for c in new:
        prompt = tokenizer.apply_chat_template([{"role": "system", "content": SHORT}, {"role": "user", "content": c["text"]}],
                                               tokenize=False, add_generation_prompt=True)
        ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids
        with torch.no_grad():
            out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=32, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        reply = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        recorded = replies(RECORDED, "new")[c["id"]]["raw"]
        print(f"{c['text'][:48]:48s} {reply}{'' if reply == recorded else '   (differs from the recording)'}")

## 10. The decision

> **Your turn 3.** Fernway's memo ends with one word: `"go"` (ship the adapter in the app) or `"no-go"` (keep the hosted system). Put your decision in `decision`, as the targets and the stop rule say. Then run the check cell.

In [ ]:
decision = "no-go"

In [ ]:
expect_hash('decision', decision, 'fc52cc1f50a1efb4')

The page's section *The recommendation* has Hana's result memo, and what would change the answer.

## Recap

- One measure, the same dev cases, targets written first: the evidence decides, not the hope.
- Prompt only (hosted): 198 of 200 dev decisions right. The best adapter: 183, with only 40 of 50 out-of-scope messages caught. The stop rule says no-go.
- More in-scope data improved the intents and not the rare class: the out-of-scope share of training fell from 12% to 7%.
- A threshold trades caught messages for false handovers; it is a patch, not a fix.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/fine-tuning/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/fine-tuning/m07/should-this-team-fine-tune/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). CLINC150 by Larson et al. (2019), CC BY 3.0. Fernway and its team are made up.